In [1]:
import tensorflow as tf

x = tf.Variable(3.0, name="x")
y = tf.Variable(4.0, name="y")

f = x*x*y + y + 2

print(f.numpy())

42.0


In [2]:

x1 = tf.Variable(1, name="x1")

print(f"Is eager execution enabled? {tf.executing_eagerly()}")
print(f"Variable value: {x1.numpy()}")

Is eager execution enabled? True
Variable value: 1


In [3]:
graph = tf.Graph()
with graph.as_default():
    x2 = tf.Variable(2)

In [4]:
x2.graph is graph

True

# TensorFlow中的线性回归

In [5]:
import numpy as np
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
m,n = housing.data.shape
housing_data_plus_bias = np.c_[np.ones((m,1)), housing.data]

X = tf.constant(housing_data_plus_bias,dtype=tf.float32,name="X")
y = tf.constant(housing.target.reshape(-1,1),dtype=tf.float32,name="y")
XT = tf.transpose(X)
theta = tf.matmul(tf.matmul(tf.linalg.inv(tf.matmul(XT,X)),XT), y)

theta_value = theta.numpy()


---

## 1. 理论基础：线性回归的数学图景

线性回归的目标是找到一组权重 $\theta$（Theta），使得预测值与真实值之间的均方误差（MSE）最小。其模型公式为：


$$\hat{y} = X \cdot \theta$$

在 TensorFlow 中，这被视为一个计算图：数据 $X$ 流入，与权重 $\theta$ 相乘，输出预测结果。

---

## 2.正规方程（Normal Equation）

它不经过“学习”，直接利用线性代数公式求出最优解：


$$\hat{\theta} = (X^T \cdot X)^{-1} \cdot X^T \cdot y$$

### 适配 TF 2.x 的代码要点：

* **不再需要**：`tf.Session()`、`tf.global_variables_initializer()`。
* **API 修正**：使用 `tf.linalg.inv` 代替 `tf.matrix_inverse`。
* **提取结果**：直接使用 `.numpy()`。

---


# 实现梯度下降

## 手工计算梯度

In [6]:
n_epochs = 1000
learning_rate = 0.01
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 对原始数据进行缩放（注意：通常只缩放特征，不缩放偏置列和目标值）
scaled_housing_data = scaler.fit_transform(housing.data)

#  加上偏置项
scaled_housing_data_plus_bias = np.c_[np.ones((m, 1)), scaled_housing_data]

X = tf.constant(scaled_housing_data_plus_bias, dtype=tf.float32, name="X")
y = tf.constant(housing.target.reshape(-1,1), dtype=tf.float32, name="y")

theta = tf.Variable(tf.random.uniform([n+1,1],-1.0,1.0),name="theta")
for epoch in range(n_epochs):
    y_pred = tf.matmul(X,theta,name = 'predictions')
    error = y_pred-y
    mse = tf.reduce_mean(tf.square(error),name = 'mse')
    gradients = 2/m*tf.matmul(tf.transpose(X),error)
    theta.assign_sub(learning_rate*gradients)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, MSE = {mse.numpy()}")
print('Final theta', theta.numpy())

Epoch 0, MSE = 7.272852420806885
Epoch 100, MSE = 0.812599241733551
Epoch 200, MSE = 0.6837261915206909
Epoch 300, MSE = 0.640724778175354
Epoch 400, MSE = 0.6103255748748779
Epoch 500, MSE = 0.5880863666534424
Epoch 600, MSE = 0.5717580318450928
Epoch 700, MSE = 0.5597397685050964
Epoch 800, MSE = 0.550870418548584
Epoch 900, MSE = 0.5443058609962463
Final theta [[ 2.0685523 ]
 [ 0.8521926 ]
 [ 0.16213721]
 [-0.23494405]
 [ 0.24820113]
 [ 0.01098673]
 [-0.04347759]
 [-0.5624172 ]
 [-0.5321571 ]]


## 使用自动微分


在刚才的代码中，我们需要手动推导梯度的数学公式：`gradients = 2/m * tf.matmul(tf.transpose(X), error)`。如果模型变得复杂（比如有 100 层神经网路），手动推导几乎是不可能的。

**自动微分（Autodiff）** 就是让 TensorFlow 解决这件事。在 TensorFlow 2.x 中，使用 **`tf.GradientTape`**（梯度磁带）。

---

## 核心概念：`tf.GradientTape`

可以把它想象成一部**“操作录像机”**：

1. **录制**：在 `with tf.GradientTape() as tape:` 块内，TensorFlow 会记录下所有对 `tf.Variable` 的操作。
2. **倒带计算**：调用 `tape.gradient()` 时，它会根据录制好的路径，利用数学里的“链式法则”自动反向计算导数。

---


# 使用优化器

In [7]:
optimizer = tf.optimizers.SGD(learning_rate = learning_rate)
with tf.GradientTape() as tape:
    # 计算预测值
    y_pred = tf.matmul(X,theta)
    # 计算损失（MSE）
    loss = tf.reduce_mean(tf.square(y_pred - y))

# 自动求导
gradients = tape.gradient(loss,[theta])

# 使用优化器更新参数
optimizer.apply_gradients(zip(gradients,[theta]))

print(f'Loss：{loss.numpy()}')

Loss：0.5394315719604492


# 给训练算法提供数据

In [8]:
@tf.function
def compute_B(A_input):
    return A_input+5
B_val_1 = compute_B(tf.constant([[1,2,3]],dtype=tf.float32))
B_val_2 = compute_B(tf.constant([[4,5,6],[7,8,9]],dtype=tf.float32))
print(B_val_1.numpy())
print(B_val_2.numpy())

[[6. 7. 8.]]
[[ 9. 10. 11.]
 [12. 13. 14.]]


In [9]:
from sklearn.preprocessing import StandardScaler
# 设置超参数
n_epochs = 10
batch_size = 512
learning_rate = 0.001

scaler = StandardScaler()
scaled_housing_data = scaler.fit_transform(housing.data)

# 准备数据
m,n = housing.data.shape
scaled_housing_data_plus_bias = np.c_[np.ones((m,1)), scaled_housing_data]
n_batches = int(np.ceil(m / batch_size))

# 初始化变量
theta = tf.Variable(tf.random.uniform([n+1,1],-1.0,1.0),name="theta")
optimizer = tf.optimizers.SGD(learning_rate = learning_rate)

# 获取小批量的函数
def get_batch(batch_index):
    start = batch_index*batch_size
    end = start+batch_size
    X_bath = scaled_housing_data_plus_bias[start:end,:]
    y_bath = housing.target.reshape(-1,1)[start:end]
    return tf.constant(X_bath,dtype = tf.float32), tf.constant(y_bath,dtype = tf.float32)

# 执行阶段
for epoch in range(n_epochs):
    for batch_index in range(n_batches):
        X_batch,y_batch = get_batch(batch_index)
        with tf.GradientTape() as tape:
            # 前向计算
            y_pred = tf.matmul(X_batch,theta)
            error = y_pred-y_batch
            loss = tf.reduce_mean(tf.square(error))

        # 自动计算梯度并更新
        gradients = tape.gradient(loss,[theta])
        optimizer.apply_gradients(zip(gradients,[theta]))
    # 打印每个epoch的进度
    if epoch%1 ==0:
        print(f'epoch{epoch},loss{loss.numpy()}')
print('Final Theta:',theta.numpy())

epoch0,loss4.091986179351807
epoch1,loss3.041787624359131
epoch2,loss2.274034023284912
epoch3,loss1.713644027709961
epoch4,loss1.3055270910263062
epoch5,loss1.0092321634292603
epoch6,loss0.7950361967086792
epoch7,loss0.6410866975784302
epoch8,loss0.53130704164505
epoch9,loss0.45386505126953125
Final Theta: [[ 1.553213  ]
 [ 0.6136061 ]
 [ 0.30560037]
 [ 0.18408272]
 [-0.00867177]
 [ 0.3827687 ]
 [-0.15621705]
 [ 0.01229555]
 [ 0.3253869 ]]


# 保存和恢复模型

In [10]:
checkpoint = tf.train.Checkpoint(theta = theta,optimizer = optimizer)

# 在训练后或训练中保存
checkpoint.save('./models/my_model.ckpt')

manager = tf.train.CheckpointManager(checkpoint,directory='./models/my_model',max_to_keep=3)
manager.save()# 自动编号保存

'./models/my_model\\ckpt-2'

# 用TensorBoard来可视化图和训练曲线

In [11]:
from datetime import datetime

now = datetime.utcnow().strftime('%Y%m%d%H%M%S')
root_logdir = 'tf_logs'
logdir = '{}/run-{}'.format(root_logdir,now)
file_writer = tf.summary.create_file_writer(logdir=logdir)

In [12]:
# 在训练循环中
for epoch in range(n_epochs):

    for batch_index in range(n_batches):
        X_batch,y_batch  = get_batch(batch_index)
        with tf.GradientTape() as tape:
            y_pred = tf.matmul(X_batch,theta)
            loss = tf.reduce_mean(tf.square(y_pred - y_batch))

        # 应用梯度
        gradients = tape.gradient(loss,[theta])
        optimizer.apply_gradients(zip(gradients,[theta]))

        if batch_index % 10 ==0:
            step = epoch * n_batches +batch_index
            with file_writer.as_default():
                tf.summary.scalar('MSE loss',loss,step=step)


In [13]:
file_writer.close()